In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# House Prices — Starter Notebook
# Paste into a Kaggle Notebook created from the competition page:
# ../input/house-prices-advanced-regression-techniques/

# 0) Imports and utilities
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings("ignore")

# RMSE helper (Kaggle uses RMSE on log or raw? competition uses RMSE on log-transformed SalePrice)
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# cross-validated RMSE for a model (using log1p transformation recommended)
def cv_rmse(model, X, y, folds=5, random_state=42):
    kf = KFold(n_splits=folds, shuffle=True, random_state=random_state)
    scores = []
    for train_idx, val_idx in kf.split(X):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        model.fit(X_train, y_train)
        preds = model.predict(X_val)
        # note: if we model log1p(SalePrice), we should inverse-transform here
        scores.append(rmse(y_val, preds))
    return np.mean(scores), np.std(scores)

# 1) Load data
train = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
test = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")
submission = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/sample_submission.csv")

print(f"train shape: {train.shape}, test shape: {test.shape}")

# Quick peek
display(train.head())
display(train[['Id','SalePrice']].describe())

# 2) Quick EDA starters
# Distribution of target
plt.figure(figsize=(8,4))
sns.histplot(train['SalePrice'], kde=True)
plt.title("SalePrice distribution (raw)")
plt.show()

# log-transform target often helps
plt.figure(figsize=(8,4))
sns.histplot(np.log1p(train['SalePrice']), kde=True)
plt.title("log1p(SalePrice) distribution")
plt.show()

# 3) Basic preprocessing strategy
# - Combine train/test for consistent preprocessing
# - Record Ids and target
train_ids = train['Id']
test_ids = test['Id']
y = np.log1p(train['SalePrice'])   # we model log1p(SalePrice)
train.drop(['Id','SalePrice'], axis=1, inplace=True)
test.drop(['Id'], axis=1, inplace=True)
all_data = pd.concat([train, test], axis=0).reset_index(drop=True)
print("Combined shape:", all_data.shape)

# 3a) Feature selection: simple starter: use features with reasonable cardinality
# We'll create lists of numeric and categorical columns
numeric_feats = all_data.select_dtypes(include=['int64','float64']).columns.tolist()
categorical_feats = all_data.select_dtypes(include=['object']).columns.tolist()
print("Numeric:", len(numeric_feats), "Categorical:", len(categorical_feats))

# 3b) A few simple engineered features (common useful ones)
all_data['TotalSF'] = all_data['TotalBsmtSF'].fillna(0) + all_data['1stFlrSF'].fillna(0) + all_data['2ndFlrSF'].fillna(0)
all_data['HouseAge'] = 2025 - all_data['YearBuilt']  # adjust year if you want current year; Kaggle uses dataset year - keep consistent
# You can add more domain-driven features later.

# update numeric/categorical lists after adding engineered features
numeric_feats = [c for c in numeric_feats if c in all_data.columns]
numeric_feats += ['TotalSF','HouseAge']
categorical_feats = [c for c in categorical_feats if c in all_data.columns]

# 4) Build preprocessing pipelines
# Numeric pipeline: impute median + scale
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical pipeline: impute constant + one-hot (drop='ignore' to handle unseen categories)
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='MISSING')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_feats),
        ('cat', categorical_transformer, categorical_feats)
    ],
    remainder='drop'  # drop any other columns
)

# 5) Prepare matrices for modeling
# Fit preprocessor on combined data (we'll split later)
X_all = preprocessor.fit_transform(all_data)
# Re-split into train/test
X_train = X_all[:len(y), :]
X_test = X_all[len(y):, :]

print("X_train shape:", X_train.shape, "X_test shape:", X_test.shape)

# 6) Models to evaluate
# We'll evaluate models on the training set with cross-validation (using log1p target)
from sklearn.base import clone

models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "RandomForest": RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    "XGBoost": xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, random_state=42, verbosity=0),
    "LightGBM": lgb.LGBMRegressor(n_estimators=1000, learning_rate=0.03, random_state=42),
    "CatBoost": CatBoostRegressor(iterations=1000, learning_rate=0.03, verbose=0, random_state=42)
}

# Cross-validate each model using KFold on X_train / y
results = []
for name, model in models.items():
    print("Training CV for:", name)
    # wrap with pipeline so that we keep preprocessing integrated (we already preprocessed separately here)
    # using preprocessed arrays directly:
    mean_rmse, std_rmse = cv_rmse(clone(model), pd.DataFrame(X_train), y.reset_index(drop=True), folds=5)
    results.append((name, mean_rmse, std_rmse))
    print(f"{name}: RMSE = {mean_rmse:.4f} ± {std_rmse:.4f}")

# summary table
res_df = pd.DataFrame(results, columns=['model','cv_rmse_mean','cv_rmse_std']).sort_values('cv_rmse_mean')
display(res_df)

# 7) Fit best model on full training set and inspect feature importances
best_model_name = res_df.iloc[0]['model']
print("Best model (by CV):", best_model_name)
best_model = models[best_model_name]
best_model.fit(X_train, y)

# Feature importance: depends on model type
def get_feature_names(preprocessor):
    # numeric features + onehot categories names
    num_names = numeric_feats
    # get ohe feature names
    cat_pipe = preprocessor.named_transformers_['cat']
    ohe = cat_pipe.named_steps['onehot']
    cat_names = ohe.get_feature_names_out(categorical_feats).tolist()
    return num_names + cat_names

feature_names = get_feature_names(preprocessor)
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    fi = pd.Series(importances, index=feature_names).sort_values(ascending=False).head(20)
    plt.figure(figsize=(8,6))
    sns.barplot(x=fi.values, y=fi.index)
    plt.title(f"Top 20 feature importances ({best_model_name})")
    plt.show()
elif hasattr(best_model, 'coef_'):
    coefs = best_model.coef_
    fi = pd.Series(coefs, index=feature_names).abs().sort_values(ascending=False).head(20)
    plt.figure(figsize=(8,6))
    sns.barplot(x=fi.values, y=fi.index)
    plt.title(f"Top 20 coefficient magnitudes ({best_model_name})")
    plt.show()
else:
    print("Model does not expose importances or coef_ easily. Consider SHAP or permutation importance.")

# 8) Create predictions and submission
preds_test_log = best_model.predict(X_test)  # modeled on log1p
preds_test = np.expm1(preds_test_log)  # inverse transform

submission_df = pd.DataFrame({
    "Id": test_ids,
    "SalePrice": preds_test
})
# Clip negatives just in case
submission_df['SalePrice'] = submission_df['SalePrice'].clip(lower=0)
display(submission_df.head())

# Save CSV for submission
submission_df.to_csv("submission.csv", index=False)
print("Saved submission.csv — upload this to the Kaggle submission page.")

# 9) Next steps & suggestions (print quick notes)
notes = """
Next steps to improve:
- More careful feature engineering (categorical cardinality reduction, domain-specific transforms)
- Target leakage check and stronger outlier handling
- Hyperparameter tuning (RandomizedSearchCV/Optuna) for top models
- Stacking/ensembling (blend LGBM + XGBoost + CatBoost)
- Use SHAP for model explainability and deeper feature selection
- Consider modeling log1p(SalePrice) (we used that here) and verifying validation pipeline
"""
print(notes)
